# Week 1 Clean up NYC mobility data

In [2]:
import duckdb

con = duckdb.connect("taxi_project")

In [3]:
con.sql("""
SELECT 1 AS TEST;
""").df()

,TEST
0,1


In [5]:
con.sql("""
CREATE TABLE yellow_taxi_raw AS
        SELECT *
        FROM '../data/nyc_taxi/trips/2025/yellow_tripdata_2025-01.parquet'
""")

In [8]:
con.sql("""
SELECT * FROM yellow_taxi_raw LIMIT 1;
""")

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┬────────────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ Airport_fee │ cbd_congestion_fee │
│  int32   │      timestamp       │       timestamp       │      int64      │    double     │   int64    │      varchar       │    int32     │    int32     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │       double       │
├──────

In [40]:
con.sql(
    """ 
    SELECT vendorID, COUNT(*)
    FROM yellow_taxi_raw
    GROUP BY vendorID
"""
)

┌──────────┬──────────────┐
│ VendorID │ count_star() │
│  int32   │    int64     │
├──────────┼──────────────┤
│        1 │       753671 │
│        2 │      2719860 │
│        6 │          489 │
│        7 │         1206 │
└──────────┴──────────────┘

In [43]:
con.sql( """
    CREATE TABLE taxi_clean AS
        SELECT
            tpep_pickup_datetime as pickup_datetime,
            tpep_dropoff_datetime as dropoff_datetime,
            passenger_count,
            trip_distance,
            PULocationID as pu_location_id,
            DOLocationID as do_location_id,
            fare_amount,
            tip_amount
        FROM
            yellow_taxi_raw
        WHERE 
            pickup_datetime IS NOT NULL AND
            dropoff_datetime IS NOT NULL AND
            passenger_count IS NOT NULL AND
            trip_distance IS NOT NULL AND
            PULocationID IS NOT NULL AND
            DOLocationID IS NOT NULL AND
            fare_amount IS NOT NULL AND
            tip_amount IS NOT NULL AND
            pickup_datetime > dropoff_datetime AND
            dropoff_datetime < pickup_datetime AND
            passenger_count > 0 AND
            trip_distance >= 0 AND
            fare_amount >= 0 AND
            tip_amount >= 0
""")

In [44]:
con.sql("""
SHOW TABLES;
""")

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ taxi_clean      │
│ yellow_taxi_raw │
└─────────────────┘

In [46]:
con.sql(
    """
    SELECT * FROM taxi_clean LIMIT 1;
"""
)

┌─────────────────────┬─────────────────────┬─────────────────┬───────────────┬────────────────┬────────────────┬─────────────┬────────────┐
│   pickup_datetime   │  dropoff_datetime   │ passenger_count │ trip_distance │ pu_location_id │ do_location_id │ fare_amount │ tip_amount │
│      timestamp      │      timestamp      │      int64      │    double     │     int32      │     int32      │   double    │   double   │
├─────────────────────┼─────────────────────┼─────────────────┼───────────────┼────────────────┼────────────────┼─────────────┼────────────┤
│ 2025-01-02 12:26:00 │ 2025-01-02 11:29:58 │               1 │           9.0 │             88 │            210 │        45.5 │        0.0 │
└─────────────────────┴─────────────────────┴─────────────────┴───────────────┴────────────────┴────────────────┴─────────────┴────────────┘